# 新12개 가스+최고미성형 구간제외 결합평가


In [ ]:
from pathlib import Path
import nbformat,numpy as np,pandas as pd,json
from sklearn.metrics import confusion_matrix
root=Path.cwd();tables=root/'reports/tables'
s=''.join(c.source for c in nbformat.read(root/'notebooks/193_rg3_fresh12_full_reproduction.ipynb',4).cells if c.cell_type=='code')
ns={'__name__':'__main__'};exec(s[:s.index('records=[];cutrows=[];first=None')],ns)
train=ns['d'].copy();n=len(train)
allshots=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
allshots=allshots[allshots['product'].eq('RG3')].copy()
missing=allshots[~allshots.TimeStamp.isin(train.TimeStamp)].copy()
assert len(missing)==8 and missing.Reason.eq('미성형').all()
ns['d']=pd.concat([train,missing],ignore_index=True)
ns['y']=np.r_[train.PassOrFail.to_numpy(int),np.full(8,-1)]
ns['groups']=ns['d'].comparison_block
ns['F']={k:pd.DataFrame(ns['physical'](ns['d'],k)) for k in ['thermal','pressure']}
cuts=pd.read_csv(tables/'rg3_193_fresh12_thresholds.csv').set_index('block')
gas=pd.read_csv(tables/'rg3_193_fresh12_predictions.csv',parse_dates=['TimeStamp'])[['TimeStamp','prediction']]
extra=[]
for block in missing.comparison_block.unique():
 ref=np.flatnonzero(~train.comparison_block.eq(block))
 q=np.flatnonzero(ns['groups'].eq(block).to_numpy()&(np.arange(len(ns['d']))>=n))
 score,z=ns['fresh_scores'](ref,q);a=ns['d'].iloc[q];c=cuts.loc[block]
 direction=((a.Max_Injection_Pressure__delta1.round(2)<0)&(a.Max_Injection_Speed__delta1.round(2)>0)&(a.Max_Switch_Over_Pressure__delta1.round(2)<a.Max_Injection_Pressure__delta1.round(2))).to_numpy()
 pr=(score['baseline']>c.baseline)|((score['expert']>c.expert)&(a.Barrel_Temperature_5__delta1.to_numpy()>c.delta))|((score['thermal']>c.thermal)&(z['thermal'][:,4]<c.thermal_gate))|((score['pressure']>c.pressure)&direction)
 extra.extend({'TimeStamp':ts,'prediction':int(p)} for ts,p in zip(a.TimeStamp,pr))
gas=pd.concat([gas,pd.DataFrame(extra)]).rename(columns={'prediction':'gas_prediction'})
short=pd.read_csv(tables/'rg3_shortshot_143_fixed_candidate_predictions.csv',parse_dates=['TimeStamp'])[['TimeStamp','prediction']].rename(columns={'prediction':'shortshot_prediction'})
joined=allshots[['TimeStamp','PassOrFail','Reason']].merge(gas,on='TimeStamp',validate='one_to_one').merge(short,on='TimeStamp',validate='one_to_one')
assert len(joined)==591 and joined.TimeStamp.is_unique
joined['any_defect_prediction']=(joined.gas_prediction.eq(1)|joined.shortshot_prediction.eq(1)).astype(int)
tn,fp,fn,tp=confusion_matrix(joined.PassOrFail,joined.any_defect_prediction,labels=[0,1]).ravel()
summary={'N':591,'defects':25,'TP':int(tp),'FP':int(fp),'FN':int(fn),'TN':int(tn),'F1':float(2*tp/(2*tp+fp+fn)),'evaluation':'development block-excluded OR; not untouched independent test'}
joined.to_csv(tables/'rg3_195_gas12_shortshot_best_OOF_predictions.csv',index=False,encoding='utf-8-sig')
(tables/'rg3_195_gas12_shortshot_best_OOF_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(summary)

